# Codebase Q&A — Step 2: Embed, Store & Ask

**Goal:**
1. Embed all chunks with a local model → store in ChromaDB
2. Test semantic search / retrieval
3. Run full RAG pipeline → get answers with citations via Groq (Llama 3)

## 0. Setup

In [ ]:
import sys
sys.path.append('..')   # project root on path

# Verify .env loaded and GROQ_API_KEY is set
from config import GROQ_API_KEY, LLM_MODEL, EMBEDDING_MODEL, CHROMA_DB_PATH
print(f'LLM model        : {LLM_MODEL}')
print(f'Embedding model  : {EMBEDDING_MODEL}')
print(f'ChromaDB path    : {CHROMA_DB_PATH}')
print(f'GROQ_API_KEY set : {bool(GROQ_API_KEY)}')

## 1. Run full ingestion pipeline (load → chunk → embed → store)

In [ ]:
from ingestion.embedder import run_ingestion_pipeline

# reset_db=True wipes existing DB and rebuilds from scratch
# reset_db=False (default) skips re-embedding already-stored chunks
collection = run_ingestion_pipeline(repo_path='../httpx', reset_db=True)
print(f'\nTotal chunks in DB: {collection.count()}')

## 2. Test semantic search (retrieval only, no LLM)

In [ ]:
from retrieval.retriever import CodebaseRetriever

retriever = CodebaseRetriever()

In [ ]:
# Basic search
results = retriever.search('how does httpx handle timeouts?', top_k=3)

for i, r in enumerate(results, 1):
    print(f'[{i}] score={r.score:.3f}  {r.format_source()}')
    print(f'     {r.text[:300].strip()}...')
    print()

In [ ]:
# Filter search — only Python files
results = retriever.search(
    'authentication and credentials',
    top_k=3,
    language='python',
)

for i, r in enumerate(results, 1):
    print(f'[{i}] score={r.score:.3f}  {r.format_source()}')
    print()

In [ ]:
# Filter search — only files in a specific path
results = retriever.search(
    'send request',
    top_k=5,
    path_prefix='httpx',   # change to a subfolder to narrow scope
)

for i, r in enumerate(results, 1):
    print(f'[{i}] score={r.score:.3f}  {r.format_source()}')

## 3. Full RAG pipeline — ask questions, get answers

In [ ]:
from generation.qa_chain import CodebaseQAChain

chain = CodebaseQAChain()

In [ ]:
# Ask a question!
result = chain.ask('How does httpx handle timeouts?', verbose=True)
result.print()

In [ ]:
result = chain.ask('What authentication methods does httpx support?')
result.print()

In [ ]:
result = chain.ask('How does AsyncClient differ from the regular Client?')
result.print()

In [ ]:
# Try your own questions!
result = chain.ask('How does httpx handle redirects?')
result.print()

## 4. Inspect retrieved context (understand what LLM sees)
Useful for debugging bad answers

In [ ]:
question = 'how are cookies handled?'
chunks = retriever.search(question, top_k=4)
context = retriever.format_context(chunks)

print(f'Context sent to LLM for: "{question}"')
print('='*60)
print(context)

## ✅ Step 2 Complete!

You now have a working end-to-end RAG system:
- ✅ Local embeddings (sentence-transformers, free)
- ✅ ChromaDB vector store (persistent, local)
- ✅ Semantic search with metadata filters
- ✅ Groq + Llama 3 for generation (free API)
- ✅ Source citations in every answer

**To use from CLI:**
```bash
python main.py index
python main.py ask
python main.py ask "How does httpx handle redirects?"
```

**Next improvements you can add:**
- 🔀 Hybrid search (semantic + BM25 keyword)
- 🔁 Re-ranking with a cross-encoder
- 🖥️ Streamlit UI
- 🔄 Auto re-index when repo changes